In [1]:
# -*- coding: utf-8 -*-
"""Example of using PCA for outlier detection
"""
# Author: Yue Zhao <zhaoy@cmu.edu>
# License: BSD 2 clause

from __future__ import division
from __future__ import print_function

import os
import sys

# temporary solution for relative imports in case pyod is not installed
# if pyod is installed, no need to use the following line
sys.path.append(
    os.path.abspath(os.path.join(os.path.dirname("__file__"), '..')))

from pyod.models.pca import PCA
from pyod.utils.data import generate_data
from pyod.utils.data import evaluate_print
import numpy as np
from pyod.utils.example import visualize

In [2]:
contamination = 0  # percentage of outliers
n_train = 100  # number of training points
n_test = 100  # number of testing points

# Generate sample data
X_train, X_test, y_train, y_test = \
    generate_data(n_train=n_train,
                  n_test=n_test,
                  n_features=20,
                  contamination=contamination,
                  random_state=42)

In [3]:
print(X_train,
      y_train)
print(X_test,
      y_test)

[[6.43365854 5.5091683  5.04469788 ... 5.86930893 4.82256361 7.18593123]
 [5.98049594 6.28356746 6.33258429 ... 5.67376352 5.64370447 7.21564822]
 [6.25760622 4.88869009 4.2626848  ... 5.16427815 4.50406714 4.90993249]
 ...
 [5.52577629 4.3268382  5.45884369 ... 6.14947541 7.08679098 5.33684146]
 [5.9149861  6.76177647 7.68309833 ... 7.04336494 5.57867481 6.80879228]
 [6.16879278 7.37648564 5.90132133 ... 7.49972786 5.42902554 6.26810209]] [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0.]
[[5.62239682 6.50110178 5.07005779 ... 4.89710414 5.44035803 6.46543689]
 [6.54068931 6.3238704  7.03739392 ... 4.7730838  6.56371728 6.74191927]
 [6.70582027 6.1667417  5.2287215  ... 6.2725435  6.06248643 5.55746596]
 ...
 [6.32076592 5.86001731 5.

In [4]:
from pyod.models.thresholds import FILTER


In [5]:
# train PCA detector
clf_name = 'PCA'
clf = PCA(n_components=5, contamination=0.1)
clf.fit(X_train)


PCA(contamination=0.1, copy=True, iterated_power='auto', n_components=5,
  n_selected_components=None, random_state=None, standardization=True,
  svd_solver='auto', tol=0.0, weighted=True, whiten=False)

In [6]:
print(clf.threshold_)

328.4007616510845


### 训练时数据的contamination=0.1，测试时数据的contamination=0.5该怎么处理？

In [7]:
# get the prediction labels and outlier scores of the training data
y_train_scores = clf.decision_scores_  # raw outlier scores
th = clf.threshold_
y_train_pred = clf.labels_  # binary labels (0: inliers, 1: outliers)

print(y_train_scores,
      '\n',th,
      '\n', y_train_pred)

[299.20038529 199.82847474 311.12950908 246.71038557 304.09561788
 286.41330985 264.35678827 221.17366918 233.71656264 338.99276905
 227.12846105 271.3770294  219.23762525 272.44948382 237.52361693
 274.07246683 325.07208869 241.18208613 310.30735879 256.44967219
 238.59464562 282.797608   314.06902032 327.27374711 269.34603792
 206.26147417 360.0836171  299.6325262  284.30508385 326.59053991
 295.02425469 359.72222397 199.88739542 261.35029517 354.40379906
 284.82283138 188.87411976 289.6024171  244.0681229  248.84247794
 232.22536431 333.49842448 271.46355884 193.13353035 265.5266663
 220.73397599 172.79366588 249.61747846 273.91395319 269.7407897
 220.31032056 336.86805023 309.62316913 342.28259009 315.15700545
 324.85956261 236.31411962 211.58148837 312.65653402 325.22660234
 223.62038963 254.752064   249.40873943 357.05711116 261.50218947
 260.31052196 327.83435467 288.4127211  276.18306665 275.35298827
 239.47986373 297.50586561 220.67023531 249.737558   260.55550738
 256.9515947

In [8]:
# get the prediction on the test data

y_test_scores = clf.decision_function(X_test)  # outlier scores
y_test_pred = clf.predict(X_test)  # outlier labels (0 or 1)
print(y_test_scores,
      y_test_pred)

[241.87888571 341.35605653 213.38823177 258.5257837  214.75548726
 200.86276286 278.20022161 267.3119599  211.14223835 321.83390002
 240.29309218 306.89076343 236.61907718 263.39867005 269.17707653
 290.17205707 343.21634516 279.6013896  203.55007809 287.20564367
 307.62716845 266.28824937 276.08671279 323.77752219 272.53022559
 300.49225527 264.55420961 270.61613667 299.30326332 286.78223407
 348.22487419 270.50976541 295.91886197 240.208547   327.40315262
 292.38477465 238.00160705 213.36877721 282.13558455 235.62037244
 261.34999913 215.84071788 361.80576554 248.8563355  221.78317706
 266.87035622 305.50223292 229.91324548 237.33175566 244.03338215
 230.38818891 276.24223324 227.11796893 286.39744524 247.33987816
 328.69210689 194.89873063 254.81738143 256.99682785 357.7146695
 310.72290534 266.23477174 239.48940807 272.77134192 256.28764026
 227.60711337 247.87264921 325.6215575  245.49028683 255.41286659
 298.68370487 260.87405612 357.27944815 209.67631636 280.67340274
 293.953094

In [9]:
clf.contamination = 0.5
clf._process_decision_scores()

PCA(contamination=0.5, copy=True, iterated_power='auto', n_components=5,
  n_selected_components=None, random_state=None, standardization=True,
  svd_solver='auto', tol=0.0, weighted=True, whiten=False)

In [10]:
print(clf.contamination)
print(clf.threshold_)

0.5
269.54341380781125


In [11]:
# get the prediction labels and outlier scores of the training data
y_train_scores = clf.decision_scores_  # raw outlier scores
th = clf.threshold_
y_train_pred = clf.labels_  # binary labels (0: inliers, 1: outliers)
print(y_train_scores,
      '\n',th,
      '\n', y_train_pred)

[299.20038529 199.82847474 311.12950908 246.71038557 304.09561788
 286.41330985 264.35678827 221.17366918 233.71656264 338.99276905
 227.12846105 271.3770294  219.23762525 272.44948382 237.52361693
 274.07246683 325.07208869 241.18208613 310.30735879 256.44967219
 238.59464562 282.797608   314.06902032 327.27374711 269.34603792
 206.26147417 360.0836171  299.6325262  284.30508385 326.59053991
 295.02425469 359.72222397 199.88739542 261.35029517 354.40379906
 284.82283138 188.87411976 289.6024171  244.0681229  248.84247794
 232.22536431 333.49842448 271.46355884 193.13353035 265.5266663
 220.73397599 172.79366588 249.61747846 273.91395319 269.7407897
 220.31032056 336.86805023 309.62316913 342.28259009 315.15700545
 324.85956261 236.31411962 211.58148837 312.65653402 325.22660234
 223.62038963 254.752064   249.40873943 357.05711116 261.50218947
 260.31052196 327.83435467 288.4127211  276.18306665 275.35298827
 239.47986373 297.50586561 220.67023531 249.737558   260.55550738
 256.9515947

In [12]:
# 修改clf.contamination后，影响clf.threshold_，不影响test的score。因此test的socre不变，但是test的labels会变。
y_test_scores = clf.decision_function(X_test)  # outlier scores
y_test_pred = clf.predict(X_test)  # outlier labels (0 or 1)
print(y_test_scores,
      y_test_pred)

[241.87888571 341.35605653 213.38823177 258.5257837  214.75548726
 200.86276286 278.20022161 267.3119599  211.14223835 321.83390002
 240.29309218 306.89076343 236.61907718 263.39867005 269.17707653
 290.17205707 343.21634516 279.6013896  203.55007809 287.20564367
 307.62716845 266.28824937 276.08671279 323.77752219 272.53022559
 300.49225527 264.55420961 270.61613667 299.30326332 286.78223407
 348.22487419 270.50976541 295.91886197 240.208547   327.40315262
 292.38477465 238.00160705 213.36877721 282.13558455 235.62037244
 261.34999913 215.84071788 361.80576554 248.8563355  221.78317706
 266.87035622 305.50223292 229.91324548 237.33175566 244.03338215
 230.38818891 276.24223324 227.11796893 286.39744524 247.33987816
 328.69210689 194.89873063 254.81738143 256.99682785 357.7146695
 310.72290534 266.23477174 239.48940807 272.77134192 256.28764026
 227.60711337 247.87264921 325.6215575  245.49028683 255.41286659
 298.68370487 260.87405612 357.27944815 209.67631636 280.67340274
 293.953094

### 修改了contamination后会改变事先存储的模型吗？

In [13]:
# train PCA detector
clf_name = 'PCA'
clf = PCA(n_components=5, contamination=0.1)
clf.fit(X_train)


PCA(contamination=0.1, copy=True, iterated_power='auto', n_components=5,
  n_selected_components=None, random_state=None, standardization=True,
  svd_solver='auto', tol=0.0, weighted=True, whiten=False)

In [14]:
print(clf.contamination)
print(clf.threshold_)

0.1
328.4007616510845


In [16]:
from joblib import dump, load
# save the model
dump(clf, 'pca.joblib')
# load the model
clf_load = load('pca.joblib')
print(clf_load)
print(clf_load.contamination)
print(clf_load.threshold_)

PCA(contamination=0.1, copy=True, iterated_power='auto', n_components=5,
  n_selected_components=None, random_state=None, standardization=True,
  svd_solver='auto', tol=0.0, weighted=True, whiten=False)
0.1
328.4007616510845


In [17]:
clf_load.contamination = 0.5
clf_load._process_decision_scores()
print(clf_load.contamination)
print(clf_load.threshold_)

0.5
269.54341380781125


In [20]:
# get the prediction labels and outlier scores of the training data
y_train_scores = clf_load.decision_scores_  # raw outlier scores
th = clf_load.threshold_
y_train_pred = clf_load.labels_  # binary labels (0: inliers, 1: outliers)
print(y_train_scores,
      '\n',th,
      '\n', y_train_pred)

[299.20038529 199.82847474 311.12950908 246.71038557 304.09561788
 286.41330985 264.35678827 221.17366918 233.71656264 338.99276905
 227.12846105 271.3770294  219.23762525 272.44948382 237.52361693
 274.07246683 325.07208869 241.18208613 310.30735879 256.44967219
 238.59464562 282.797608   314.06902032 327.27374711 269.34603792
 206.26147417 360.0836171  299.6325262  284.30508385 326.59053991
 295.02425469 359.72222397 199.88739542 261.35029517 354.40379906
 284.82283138 188.87411976 289.6024171  244.0681229  248.84247794
 232.22536431 333.49842448 271.46355884 193.13353035 265.5266663
 220.73397599 172.79366588 249.61747846 273.91395319 269.7407897
 220.31032056 336.86805023 309.62316913 342.28259009 315.15700545
 324.85956261 236.31411962 211.58148837 312.65653402 325.22660234
 223.62038963 254.752064   249.40873943 357.05711116 261.50218947
 260.31052196 327.83435467 288.4127211  276.18306665 275.35298827
 239.47986373 297.50586561 220.67023531 249.737558   260.55550738
 256.9515947

In [21]:
# 修改clf.contamination后，影响clf.threshold_，不影响test的score。因此test的socre不变，但是test的labels会变。
y_test_scores = clf_load.decision_function(X_test)  # outlier scores
y_test_pred = clf_load.predict(X_test)  # outlier labels (0 or 1)
print(y_test_scores,
      y_test_pred)

[241.87888571 341.35605653 213.38823177 258.5257837  214.75548726
 200.86276286 278.20022161 267.3119599  211.14223835 321.83390002
 240.29309218 306.89076343 236.61907718 263.39867005 269.17707653
 290.17205707 343.21634516 279.6013896  203.55007809 287.20564367
 307.62716845 266.28824937 276.08671279 323.77752219 272.53022559
 300.49225527 264.55420961 270.61613667 299.30326332 286.78223407
 348.22487419 270.50976541 295.91886197 240.208547   327.40315262
 292.38477465 238.00160705 213.36877721 282.13558455 235.62037244
 261.34999913 215.84071788 361.80576554 248.8563355  221.78317706
 266.87035622 305.50223292 229.91324548 237.33175566 244.03338215
 230.38818891 276.24223324 227.11796893 286.39744524 247.33987816
 328.69210689 194.89873063 254.81738143 256.99682785 357.7146695
 310.72290534 266.23477174 239.48940807 272.77134192 256.28764026
 227.60711337 247.87264921 325.6215575  245.49028683 255.41286659
 298.68370487 260.87405612 357.27944815 209.67631636 280.67340274
 293.953094

In [22]:
# 验证模型是否被修改
clf_load_ori = load('pca.joblib')
print(clf_load_ori)
print(clf_load_ori.contamination)
print(clf_load_ori.threshold_)
# 原模型没有被修改

PCA(contamination=0.1, copy=True, iterated_power='auto', n_components=5,
  n_selected_components=None, random_state=None, standardization=True,
  svd_solver='auto', tol=0.0, weighted=True, whiten=False)
0.1
328.4007616510845
